# PINN Single-Factor Ablation Study (Colab GPU Runner)

This notebook executes the systematic single-factor PINN ablation study around the frozen Exp D baseline.

### Ablation Matrix (3 seeds: 42, 43, 44 each):
1. **Activation:** `tanh` (base) vs `GELU` vs `sin`
2. **Collocation:** `base` (resampled uniform) vs `fixed` (static uniform) vs `sobol` vs `lhs`
3. **Loss Weighting:** `base` (pure PDE) vs `data_equal` (PDE + data) vs `data_heavy` vs `grad_norm` (dynamic GradNorm balancing)
4. **Optimizer:** `base` (Adam + L-BFGS) vs `adam_only` (Adam 1500 epochs)

### Prerequisites:
1. In Google Colab menu: **Runtime -> Change runtime type -> GPU (T4 or better)**.
2. Have `pinn_ablation_colab_package.zip` ready on your computer.

In [ ]:
# Step 1: Check GPU availability
!nvidia-smi

In [ ]:
# Step 2: Upload and extract pinn_ablation_colab_package.zip
import os, zipfile
from google.colab import files

if not os.path.exists('pinn_ablation_colab_package.zip'):
    print('Please select and upload pinn_ablation_colab_package.zip from your computer:')
    uploaded = files.upload()

print('Extracting package...')
with zipfile.ZipFile('pinn_ablation_colab_package.zip', 'r') as z:
    z.extractall('.')
print('Package extracted successfully!')
!ls -lh data/datasets/

In [ ]:
# Step 3: Fast 2-epoch smoke test to verify GPU, autograd, Sobol, LHS, GradNorm
!python experiments/train_pinn_ablation.py --device cuda --smoke-test --out-dir /tmp/pinn_smoke

In [ ]:
# Step 4: Run full PINN ablation study (all 12 configurations x 3 seeds = 36 runs)
# Results are incrementally saved to data/results/pinn_ablation/ with resume support.
!python experiments/train_pinn_ablation.py --device cuda --seeds 42,43,44 --epochs 1000 --lbfgs-iters 500 --out-dir data/results/pinn_ablation --resume

In [ ]:
# Step 5: Display summary table
import pandas as pd
df = pd.read_csv('data/results/pinn_ablation/pinn_ablation_summary.csv')
print(df.to_string())

# Group by variant (mean across seeds)
summary_tbl = df.groupby(['axis', 'variant'])[['interp_rel_l2_median', 'low_alpha_ood_rel_l2_median', 'high_alpha_ood_rel_l2_median', 't0_mae_median', 'train_time_sec']].mean()
print('\n=== MEAN METRICS ACROSS SEEDS (42, 43, 44) ===')
print(summary_tbl.to_string())

In [ ]:
# Step 6: Create downloadable ZIP archive of all results, checkpoints, and curves
!zip -r pinn_ablation_results.zip data/results/pinn_ablation/
print('Results successfully packaged into pinn_ablation_results.zip!')
from google.colab import files
files.download('pinn_ablation_results.zip')

In [ ]:
# Step 7 (Optional): Backup to Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    !mkdir -p /content/drive/MyDrive/Digital_Twin_Ablation/
    !cp -r data/results/pinn_ablation /content/drive/MyDrive/Digital_Twin_Ablation/
    print('Backed up results to Google Drive successfully!')
except Exception as e:
    print('Google Drive backup skipped:', e)